# Basics

The shortest path to a probability, and the conventions everything else in these notebooks assumes.

**NuOscProbExact** computes oscillation probabilities *exactly*, with no approximation beyond floating-point round-off, for any time-independent Hamiltonian. The method expands the Hamiltonian and the evolution operator in the SU(2) and SU(3) bases, which gives closed forms rather than a numerical integration.

In [1]:
import sys
import os

# Works whether or not the package is installed: the notebooks sit one level
# below the repository root, as the worked examples in test/ do.
sys.path.insert(0, os.path.abspath(os.path.join('..', 'src')))

import numpy as np
import matplotlib.pyplot as plt

import globaldefs as gd
import oscprob2nu
import oscprob3nu
import hamiltonians2nu
import hamiltonians3nu

%matplotlib inline
plt.rcParams.update({'figure.figsize': (7.2, 4.2), 'figure.dpi': 90,
                     'axes.grid': True, 'grid.alpha': 0.3,
                     'font.size': 11, 'legend.frameon': False})

# The three-flavor vacuum Hamiltonian at the NuFit best fit, normal ordering.
# It is energy-independent; dividing by the energy is what the builders do.
H_VAC_3NU = hamiltonians3nu.hamiltonian_3nu_vacuum_energy_independent(
    gd.S12_NO_BF, gd.S23_NO_BF, gd.S13_NO_BF, gd.DCP_NO_BF,
    gd.D21_NO_BF, gd.D31_NO_BF)

KM = gd.CONV_KM_TO_INV_EV      # multiply a length in km to get eV^-1
GEV = 1.0e9                    # multiply an energy in GeV to get eV

## Units

The library is unit-agnostic: it asks only that the Hamiltonian and the baseline be given in reciprocal units, so that $HL$ is dimensionless. Everything below uses **eV** for energies and **eV$^{-1}$** for baselines, with `globaldefs` supplying the conversions.

In [2]:
print("1 km          = %.4e eV^-1" % KM)
print("1 GeV         = %.0e eV" % GEV)
print("crust V_CC    = %.4e eV" % gd.VCC_EARTH_CRUST)

1 km          = 5.0677e+09 eV^-1
1 GeV         = 1e+09 eV
crust V_CC    = 1.1356e-13 eV


## One three-flavor probability

A 1 GeV neutrino travelling 1300 km in vacuum — roughly the DUNE baseline. The nine probabilities come back with the *initial* flavor varying slowest.

In [3]:
energy = 1.0*GEV
baseline = 1300.0*KM

# The vacuum Hamiltonian at this energy
H = np.asarray(H_VAC_3NU)/energy

prob = oscprob3nu.probabilities_3nu(H, baseline)

labels = ["ee", "emu", "etau", "mue", "mumu", "mutau",
          "taue", "taumu", "tautau"]
for name, value in zip(labels, prob):
    print("P_%-7s = %.6f" % (name, value))

P_ee      = 0.927678
P_emu     = 0.014323
P_etau    = 0.057999
P_mue     = 0.040227
P_mumu    = 0.378872
P_mutau   = 0.580901
P_taue    = 0.032095
P_taumu   = 0.606804
P_tautau  = 0.361100


Each initial flavor conserves probability, which is the first thing to check of any oscillation code:

In [4]:
prob = np.array(prob)
for start, flavor in zip((0, 3, 6), ("e", "mu", "tau")):
    print("sum over final flavors, from nu_%-4s = %.15f"
          % (flavor, prob[start:start+3].sum()))

sum over final flavors, from nu_e    = 1.000000000000002
sum over final flavors, from nu_mu   = 1.000000000000000
sum over final flavors, from nu_tau  = 1.000000000000000


## Two flavors

The same interface, with four probabilities instead of nine.

In [5]:
th12 = np.arcsin(np.sqrt(0.310))
H2_vac = hamiltonians2nu.hamiltonian_2nu_vacuum_energy_independent(
    th12, gd.D21_NO_BF)

Pee, Pem, Pme, Pmm = oscprob2nu.probabilities_2nu(
    np.asarray(H2_vac)/energy, baseline)
print("Pee = %.6f   Pem = %.6f" % (Pee, Pem))

Pee = 0.986609   Pem = 0.013391


## Any Hermitian matrix

Nothing above is special. The core routines take an arbitrary Hermitian matrix, which is what makes the library useful for non-standard scenarios: new interactions, Lorentz-invariance violation and sterile-like perturbations are all just entries in that matrix.

In [6]:
H_arbitrary = np.array([[0.0, 1.0+2.0j, 0.5],
                        [1.0-2.0j, 1.0, 0.0],
                        [0.5, 0.0, -1.0]], dtype=complex)

print(oscprob3nu.probabilities_3nu(H_arbitrary, 1.0))

(0.4669815815062982, 0.47912070045447136, 0.05389771803923255, 0.4791207004544712, 0.4163264072864637, 0.10455289225906624, 0.05389771803923255, 0.10455289225906628, 0.8415493897016997)


## Pass arrays, do not loop

The single most useful thing to know about performance here: the routines broadcast. A stack of Hamiltonians, an array of baselines, or both, are evaluated in one pass — tens of times faster than the equivalent Python loop, and the results are identical.

In [7]:
baselines = np.linspace(1.0, 3000.0, 5)*KM
prob_scan = oscprob3nu.probabilities_3nu(H, baselines)

print("shape:", np.shape(prob_scan))
print("P_ee along the scan: ", np.round(prob_scan[:, 0], 4))
print("all nine at point 0:", np.round(prob_scan[0], 4))

shape: (5, 9)
P_ee along the scan:  [1.     0.9542 0.8969 0.9149 0.9351]
all nine at point 0: [1. 0. 0. 0. 1. 0. 0. 0. 1.]
